In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Titanic - Machine Learning from Disaster

Dataset downloaded from this [link](https://www.kaggle.com/competitions/titanic).

In [ ]:
train_df = pd.read_csv('/kaggle/input/titanic/train.csv')

In [ ]:
train_df.head()

## Exploratory Data Analysis

This dataset contains the information about the passengers information voyaging in the great **"Titanic"**.

Objective: Classify whether a passenger survived or not based on the given information (_features_) about the passenger.

In [ ]:
import ydata_profiling

In [ ]:
report = ydata_profiling.ProfileReport(train_df)

In [ ]:
report

**Data Dictionary**

| Variable | Definition | Key |
| --- | --- | ------ | 
| survival | Survival | 0 = No, 1 = Yes |
| pclass | Ticket class | 1 = 1st, 2 = 2nd, 3 = 3rd |
| sex | Sex | |	
| Age | Age in years |
| sibsp | # of siblings / spouses aboard the Titanic |
| parch | # of parents / children aboard the Titanic |
| ticket | Ticket number |
| fare | Passenger fare |
| cabin |	Cabin number |
| embarked | Port of Embarkation | C = Cherbourg, Q = Queenstown, S = Southampton |

**Variable Notes:**

**pclass**: A proxy for socio-economic status (SES)
1st = Upper
2nd = Middle
3rd = Lower

**age**: Age is fractional if less than 1. If the age is estimated, is it in the form of xx.5

**sibsp**: The dataset defines family relations in this way...
Sibling = brother, sister, stepbrother, stepsister
Spouse = husband, wife (mistresses and fiancés were ignored)

**parch**: The dataset defines family relations in this way...
Parent = mother, father
Child = daughter, son, stepdaughter, stepson
Some children travelled only with a nanny, therefore parch=0 for them.



**Observations**:
1. No. of rows = 891
2. We have unique values for `PassengerId` and `Name`.
3. We have missing values in the foll. 3 col's:
    - Age: 177 missing
    - Cabin: 687 missing
    - Embarked: 2 missing
4. `Survived` is the target column containing whether the passenger survived or not. (`0 -> not survived` & `1 -> survived`)
5. No. of survived = 342 and No. of not survived = 549
6. `pclass` contains 55.1% belonging to class 3, 24.2% to class 1 and 20.7% to class 2.
7. There are 64.8% males and 35.2% females.
8. The oldest passenger is 80 yrs old.
9. `parch` col. suggests that 76.1% do not have parents or children aboard the titanic.
10. `sibsp` col. suggests that 68.2% do not have spouses or siblings aboard.
11. `fare` is highly skewed.
12. `embarked` col. tells that 72.4% of passengers embarked on Southampton port, 18.9% on Cherbourg port and 8.7% on Queenstown port.
13. As the `age` increases the passengers seem to have less siblings or spouses.
14. People around `age` 40 seem to pay higher fare.
15. `fare` seems to be correlated with `survival`.

In [ ]:
import seaborn as sns

In [ ]:
# sibsp vs survived
p = sns.catplot(x='SibSp', y='Survived', data=train_df, kind='bar', height=6, palette='muted')
p.set_ylabels('Survival Probability')

This plot shows that passengers having a lot of siblings/spouses have less chances of survival. While those having less siblings/spouses have more chance to survive.

In [ ]:
# Parch vs Survived
p = sns.catplot(x='Parch', y='Survived', data=train_df, kind='bar', height=6, palette='muted')
p.set_ylabels('Survival Probability')

Small families have more chance to survive.

In [ ]:
# Age vs Survived
p = sns.FacetGrid(train_df, col='Survived')
p.map(sns.histplot, 'Age', kde=True)

This plot shows us that more young passengers have more chance to survive.

In [ ]:
p = sns.barplot(x='Sex', y='Survived', data=train_df)
p.set_ylabel("Survival Probability")

It is visible from the plot that females have more chance to survive than males. This column can play an important role in the prediction of survival col.

In [ ]:
# pclass vs survived by sex
p = sns.catplot(x='Pclass', y='Survived', hue='Sex', data=train_df, height=6, kind='bar', palette='muted')

It is clear from the plot that people belonging to First class have more chance to survival than other classes.

In [ ]:
# embarked vs survived
p = sns.catplot(x='Embarked', y='Survived', data=train_df, height=6, kind='bar', palette='muted')

Passengers coming from Cherbourg (C) port have more chance of survival.

## Dealing with Missing Values

In [ ]:
test_df = pd.read_csv("/kaggle/input/titanic/test.csv")

In [ ]:
IDtest = test_df['PassengerId']
dataset = pd.concat(objs=[train_df, test_df], axis=0).reset_index(drop=True)
dataset.head()

In [ ]:
# Checking for missing values in the entire combined dataset and then performing Data Imputation
dataset = dataset.fillna(np.nan)
dataset.isnull().sum()

There are missing values in col's `Age`, `Cabin`, `Fare` and `Embarked`.

1. `Cabin` has > 1000 missing. So drop it.
2. For the other col's we'll perform the imputation.

The missing values of `Survived` are from the testset.

In [ ]:
dataset = dataset.drop(['Cabin'], axis=1)

# Doing the median imputation for fare
dataset['Fare'] = dataset['Fare'].fillna(dataset['Fare'].median())
# Doing the mode imputation for embarked
dataset['Embarked'] = dataset['Embarked'].fillna('S')

`Age` contains 263 missing values.
Since there are so many ppl, with different ages, we can't just assign some 4 yr old kid with `mean` of age. So we have to impute the missing value in the age using some other method.

We can check the title of the person with missing age and impute the missing age with the median value of all the other passengers with the same title.

We will be using regular expression to extract the title from the Name of the passenger.

In [ ]:
dataset['Title'] = dataset['Name'].str.extract('([A-Za-z]+)\.', expand=True)
dataset['Title'].value_counts()

In [ ]:
# Replace the rare title with more common ones
mapping = {'Mlle': 'Miss', 'Major': 'Mr', "Col": 'Mr', 'Sir': 'Mr', 'Don': 'Mr', 'Mme': 'Mrs', 'Jonkheer': 'Mr', 'Lady': 'Mrs', 'Capt': 'Mr', 'Countess': 'Mrs', 'Ms': 'Miss', 'Dona': 'Mrs'}
dataset.replace({'Title': mapping}, inplace=True)

In [ ]:
dataset['Title'].value_counts()

In [ ]:
# Impute missing Age values using median of Title groups
title_ages = dict(dataset.groupby('Title')['Age'].median())

# Create a col of avg ages
dataset['Age'] = dataset['Age'].fillna(dataset['Title'].map(title_ages))

In [ ]:
dataset.isna().sum()

Now we can see that there are no missing values present in the dataset.

## Feature Engineering

In [ ]:
dataset.drop(['PassengerId', 'Name'], inplace=True, axis=1)
dataset.head()

In [ ]:
# Mapping the values to 0 and 1
dataset['Sex'] = dataset['Sex'].map({'male': 0, 'female': 1})

# No. of ppl in a family including himself
dataset['Family_Size'] = dataset['SibSp'] + dataset['Parch'] + 1

# One hot encoding the Title and Embarked column
dataset = pd.get_dummies(dataset, columns=['Title'])
dataset = pd.get_dummies(dataset, columns=['Embarked'], prefix='Em')

# Converting the PClass into category type
dataset['Pclass'] = dataset['Pclass'].astype('category')
# One hot encoding the PClass column
dataset = pd.get_dummies(dataset, columns=['Pclass'], prefix='Pc')

# Applying the log tranform on Fare col. as it is skewed
dataset['Fare'] = dataset['Fare'].map(lambda i: np.log(i) if i > 0 else 0)

In [ ]:
# Extracting the ticket prefix. When there is no prefix it return X
Ticket = []
for i in list(dataset.Ticket):
    if not i.isdigit():
        Ticket.append(i.replace('.', '').replace('/', '').strip().split(' ')[0])  # Take Prefix
    else:
        Ticket.append('X')

dataset['Ticket'] = Ticket
dataset = pd.get_dummies(dataset, columns=['Ticket'], prefix='T')
dataset.head()

## Modelling

In [ ]:
# Separate train and test dataset
train_len = len(train_df)
training_dataset = dataset[:train_len]

testing_dataset = dataset[train_len:].copy()
testing_dataset.drop(labels='Survived', axis=1, inplace=True)

In [ ]:
from sklearn.model_selection import train_test_split

# Splitting the training datasets for training and testing the model
training_dataset['Survived'] = training_dataset['Survived'].astype(int)

X_train, X_test = train_test_split(training_dataset, test_size=0.3, random_state=0, stratify=training_dataset['Survived'])
y_train = X_train[['Survived']]
y_test = X_test[['Survived']]

### Support Vector Machine

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

model_svm = SVC(kernel='rbf', C=1, gamma=0.1)
model_svm.fit(X_train.drop(['Survived'], axis=1), y_train.values.ravel())
prediction1 = model_svm.predict(X_test.drop(['Survived'], axis=1))
print(f'Accuracy for rbf SVM = {accuracy_score(prediction1, y_test.values.ravel())}')

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

model_lg = LogisticRegression(solver='liblinear')
model_lg.fit(X_train.drop(['Survived'], axis=1), y_train.values.ravel())
prediction2 = model_lg.predict(X_test.drop(['Survived'], axis=1))
print(f'Accuracy for Logistic Regression = {accuracy_score(prediction2, y_test.values.ravel())}')

### Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model_dt = DecisionTreeClassifier()
model_dt.fit(X_train.drop(['Survived'], axis=1), y_train.values.ravel())
prediction3 = model_dt.predict(X_test.drop(['Survived'], axis=1))
print(f'Accuracy for Decision Tree = {accuracy_score(prediction3, y_test.values.ravel())}')

### K Nearest Neighbours Classifier

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model_knn = KNeighborsClassifier()
model_knn.fit(X_train.drop(['Survived'], axis=1), y_train.values.ravel())
prediction4 = model_knn.predict(X_test.drop(['Survived'], axis=1))
print(f'Accuracy for KNN = {accuracy_score(prediction4, y_test.values.ravel())}')

### Gaussian Naive Bayes Classifier

In [ ]:
from sklearn.naive_bayes import GaussianNB

model_gn = GaussianNB()
model_gn.fit(X_train.drop(['Survived'], axis=1), y_train.values.ravel())
prediction5 = model_gn.predict(X_test.drop(['Survived'], axis=1))
print(f'Accuracy for Naive Bayes = {accuracy_score(prediction5, y_test.values.ravel())}')

### Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(n_estimators=100)
model_rf.fit(X_train.drop(['Survived'], axis=1), y_train.values.ravel())
prediction6 = model_rf.predict(X_test.drop(['Survived'], axis=1))
print(f'Accuracy for Random Forest = {accuracy_score(prediction6, y_test.values.ravel())}')

## Cross Validation

In [ ]:
import copy
from tqdm import tqdm
from sklearn.model_selection import KFold #for K-fold cross validation
from sklearn.model_selection import cross_val_score #score evaluation
from sklearn.model_selection import cross_val_predict #prediction

kfold = KFold(n_splits=10, shuffle=True, random_state=22)
mean_of_cross_validation=list()
accuracy = list()

std_of_cross_validation = list()

classifiers = ['Radial SVM', 'Logistic Regression', 'KNN', 'Decision Tree', 'Naive Bayes', 'Random Forest']

models = [SVC(kernel='rbf'), LogisticRegression(solver='liblinear'), KNeighborsClassifier(n_neighbors=9), DecisionTreeClassifier(), GaussianNB(), RandomForestClassifier(n_estimators=100)]

for i in tqdm(models):
    model = i
    cv_result = cross_val_score(model, training_dataset.drop(['Survived'], axis=1), training_dataset['Survived'], cv=kfold, scoring='accuracy')
    mean_of_cross_validation.append(cv_result.mean())
    std_of_cross_validation.append(cv_result.std())
    accuracy.append(copy.deepcopy(cv_result))

new_models_dataframe = pd.DataFrame({'CV Mean': mean_of_cross_validation, 'Std': std_of_cross_validation}, index=classifiers)
new_models_dataframe

In [ ]:
import matplotlib.pyplot as plt
new_models_dataframe['CV Mean'].plot.barh(width=0.8)
plt.title('Average CV Mean Accuracy')
fig = plt.gcf()
fig.set_size_inches(8, 5)
plt.show()

## Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

X = training_dataset.drop(['Survived'], axis=1)
Y = training_dataset[['Survived']]

f, ax = plt.subplots(3, 2, figsize=(12, 10))

y_pred = cross_val_predict(SVC(kernel='rbf'), X, Y.values.ravel(), cv=10)
sns.heatmap(confusion_matrix(Y, y_pred), ax=ax[0, 0], annot=True, fmt='2.0f')
ax[0, 0].set_title('Matrix for rbf-SVM')

y_pred = cross_val_predict(KNeighborsClassifier(n_neighbors=9),X,Y.values.ravel(),cv=10)
sns.heatmap(confusion_matrix(Y,y_pred),ax=ax[0,1],annot=True,fmt='2.0f')
ax[0,1].set_title('Matrix for KNN')

y_pred = cross_val_predict(RandomForestClassifier(n_estimators=100),X,Y.values.ravel(),cv=10)
sns.heatmap(confusion_matrix(Y,y_pred),ax=ax[1,0],annot=True,fmt='2.0f')
ax[1,0].set_title('Matrix for Random-Forests')

y_pred = cross_val_predict(LogisticRegression(solver="liblinear"),X,Y.values.ravel(),cv=10)
sns.heatmap(confusion_matrix(Y,y_pred),ax=ax[1,1],annot=True,fmt='2.0f')
ax[1,1].set_title('Matrix for Logistic Regression')

y_pred = cross_val_predict(DecisionTreeClassifier(),X,Y.values.ravel(),cv=10)
sns.heatmap(confusion_matrix(Y,y_pred),ax=ax[2,0],annot=True,fmt='2.0f')
ax[2,0].set_title('Matrix for Decision Tree')

y_pred = cross_val_predict(GaussianNB(),X,Y.values.ravel(),cv=10)
sns.heatmap(confusion_matrix(Y,y_pred),ax=ax[2,1],annot=True,fmt='2.0f')
ax[2,1].set_title('Matrix for Naive Bayes')

plt.subplots_adjust(hspace=0.2, wspace=0.2)
plt.show()

### Conclusion
---
- The accurace of Random Forest, Logistic Regression, KNN and Decision Forest  comes out to be amazing.
- The cross validation mean for the training dataset comes out to be great for Logistic Regression and Random Forest, followed by KNN and Decision Tree.
- By looking at the metrics, we can say that rbf-SVM has a higher chance in correctly predicting dead passengers but NaiveBayes has a higher chance in correctly predicting passengers who survived.

## Hyper-Parameter Optimization

Here we'll try hyper parameter optimization for SVM and Random Forest which are showing good performance on traning dataset.

We'll use GridSearchCV for this.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Possinle hyper-parameter values for SVM
C=[0.05,0.1,0.2,0.3,0.25,0.4,0.5,0.6,0.7,0.8,0.9,1]
gamma=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]
kernel=['rbf','linear']
hyper = {'kernel': kernel, 'C': C, 'gamma': gamma}

gd = GridSearchCV(estimator=SVC(), param_grid=hyper, verbose=True)
gd.fit(X, Y.to_numpy().ravel())
print(gd.best_score_)
print(gd.best_estimator_)

In [ ]:
# Possible Hyperparameters for Random Forest
n_estimators = range(100, 1000, 100)

hyper = {'n_estimators': n_estimators}
gd = GridSearchCV(estimator=RandomForestClassifier(random_state=0), param_grid=hyper, verbose=True)
gd.fit(X, Y.to_numpy().ravel())
print(gd.best_score_)
print(gd.best_estimator_)

The best model for SVM is having accuracy 82% with C=1, gamma=0.1 and kernel='linear'

The best model for Random Forest is having accuracy 81% with n_estimators=800

## Submission on Kaggle Competition

Here we'll predict the test dataset provided in the competition. And then submitting it to competition. We'll use SVM, Logistic Regression and Random Forest models to do submission.

In [ ]:
svm_predictions = model_svm.predict(testing_dataset)
logistic_regression_predictions = model_lg.predict(testing_dataset)
random_forest_predictions = model_rf.predict(testing_dataset)

svm_submission = pd.DataFrame(
    {
        'PassengerId': IDtest,
        'Survived': svm_predictions
    }
)
svm_submission = svm_submission.reset_index(drop=True)

lg_submission = pd.DataFrame(
    {
        'PassengerId': IDtest,
        'Survived': logistic_regression_predictions
    }
)
lg_submission = lg_submission.reset_index(drop=True)

rf_submission = pd.DataFrame(
    {
        'PassengerId': IDtest,
        'Survived': random_forest_predictions
    }
)
rf_submission = rf_submission.reset_index(drop=True)

In [ ]:
import base64
from IPython.display import HTML

def create_downloadable_link(df, title="Download CSV File", filename='data.csv'):
    csv = df.to_csv(index=False)
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = f'<a download="{filename}" href="data:text/scv;base64,{payload}" target="_blank">{title} </a>'
    return HTML(html)

In [ ]:
create_downloadable_link(svm_submission.reset_index(drop=True), title="Download Support Vector Machine Submission", filename="svm_submission.csv")

In [ ]:
create_downloadable_link(lg_submission, title="Download Logistic Regression Submission", filename="lg_submission.csv")

In [ ]:
create_downloadable_link(rf_submission, title="Download Random Forest Submission", filename="rf_submission.csv")

### Accuracies on Public Leaderboard:
---
Logistic Regression: 0.76794

SVM: 0.76076

Random Forest: 0.73205

## Conclusion
---

We can see that logistic regression is performing well on public leaderboard.